## IMPORTS & FUNCTIONS

In [ ]:
import os
import glob
from pathlib import Path
import numpy as np
import tifffile
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import xarray as xr

from flopa.io.ptuio.reader import TTTRReader
from flopa.io.ptuio.reconstructor import ScanConfig, ImageReconstructor
from flopa.io.ptuio.utils import estimate_tcspc_bins, create_FLIM_image, aggregate_dataset
from flopa.io.ptuio.decoder import T3OverflowCorrector

def process_ptu(file_path, manual_res=None):
    """Loads and reconstructs a single PTU file."""
    file_name = os.path.basename(file_path).replace('.ptu', '')

    reader = TTTRReader(str(file_path))
    header_tags = reader.header.tags

    header_res = header_tags.get("MeasDesc_Resolution", 5e-12)
    tcspc_res = manual_res if manual_res is not None else header_res
    if manual_res is not None:
        sync_rate = header_tags.get("TTResult_SyncRate", 40e6)
        calculated_bins = int(np.ceil(1 / tcspc_res / sync_rate))
        buffer = 10
        tcspc_channels = calculated_bins + buffer
    else:
        tcspc_channels = estimate_tcspc_bins(header_tags)
    wrap = reader.header.tags.get("TTResultFormat_WrapAround", 1024)
    
    # Build ScanConfig dynamically based on BIDIRECTIONAL setting
    scan_args = {
        'lines': LINES,
        'pixels': PIXELS,
        'line_accumulations': LINE_ACCUMULATIONS,
        'bidirectional': BIDIRECTIONAL,
        'max_detector': MAX_DETECTOR
    }
    # Add phase shift only if bidirectional is True
    if BIDIRECTIONAL:
        scan_args['bidirectional_phase_shift'] = PHASE_SHIFT
    
    cfg = ScanConfig(**scan_args)
    
    reconstructor = ImageReconstructor(
        config=cfg, 
        tcspc_channels=tcspc_channels,
        outputs=['photon_count', 'mean_arrival_time']
    )
    corrector = T3OverflowCorrector(wraparound=wrap)

    for chunk in reader.iter_chunks():
        reconstructor.update(corrector.correct(chunk))
    
    return reconstructor.finalize(), tcspc_res, file_name

def save_outputs(ds, tcspc_res, out_dir, file_name):
    """Handles scaling, aggregation, and saving."""
    os.makedirs(out_dir, exist_ok=True)

    # 1. Scale Intensity based on Aggregation
    scale = 1
    dim_names = ['frame', 'sequence', 'detector']
    for i, flag in enumerate(AGG_CONFIG):
        if flag: scale *= ds.sizes[dim_names[i]]
    effective_int_max = INT_MAX * scale

    # 2. Aggregate
    dims_to_agg = [dim_names[i] for i, flag in enumerate(AGG_CONFIG) if flag]
    if dims_to_agg:
        ds = aggregate_dataset(ds, dims_to_agg)

    # 3. Save Loop
    for f in range(ds.sizes['frame']):
        for s in range(ds.sizes['sequence']):
            for d in range(ds.sizes['detector']):
                subset = ds.isel(frame=f, sequence=s, detector=d)
                
                # Metadata for names
                f_idx, s_idx, d_idx = int(subset.frame), int(subset.sequence), int(subset.detector)
                base_name = f"f{f_idx:03d}_s{s_idx:02d}_d{d_idx}"

                # Calculate lifetime
                intensity = subset.photon_count.values
                mean_tau_ns = subset.mean_arrival_time.values * tcspc_res * 1e9

                # --- Save TIF (Intensity) ---
                tifffile.imwrite(os.path.join(out_dir, f"{file_name}_{base_name}_int.tif"), intensity.astype(np.uint16))

                # --- Save FLIM PNG ---
                flim_rgb = create_FLIM_image(
                    mean_photon_arrival_time=mean_tau_ns,
                    intensity=intensity,
                    lt_min=LT_MIN, lt_max=LT_MAX,
                    int_min=INT_MIN, int_max=effective_int_max,
                    colormap=COLORMAP
                )
                flim_rgb = np.nan_to_num(np.clip(flim_rgb, 0, 1), nan=0.0)
                plt.imsave(os.path.join(out_dir, f"{file_name}_{base_name}_flim.png"), flim_rgb)

## USER CONFIGURATION

In [ ]:
# --- FOLDERS ---
INPUT_FOLDER = "../test_data/batch" 

# --- REQUIRED SCAN PARAMETERS (No Defaults) ---
LINES = 1024
PIXELS = 1024
LINE_ACCUMULATIONS = (16,)

# --- OPTIONAL SCAN PARAMETERS ---
BIDIRECTIONAL = True
PHASE_SHIFT = 0.00666  # Only used if BIDIRECTIONAL is True
MAX_DETECTOR = 2
TCSPC_RESOLUTION = None

# --- FLIM DATA & PROCESSING ---
LT_MIN, LT_MAX = 1.0, 3.0
INT_MIN, INT_MAX = 0, 5
AGG_CONFIG = (False, True, False) # (Frames, Sequences, Detectors)


# --- COLOR CODING ---
# Options: 
# 1. Standard name: "rainbow", "viridis", "magma", "jet"
# 2. Custom list: ["black", "green", "red"] or ["blue", "white"]
COLORMAP_SPEC = 'jet'

## BATCH EXECUTION

In [113]:
if isinstance(COLORMAP_SPEC, str):
    COLORMAP = plt.get_cmap(COLORMAP_SPEC)
else:
    COLORMAP = mcolors.LinearSegmentedColormap.from_list("custom_flim", COLORMAP_SPEC)


input_path = Path(INPUT_FOLDER)
output_path = input_path / 'reconstructed_images'

ptu_files = list(input_path.glob("*.ptu"))
print(f"Found {len(ptu_files)} files. \nStarting batch processing...")

for p_file in ptu_files:
    print(f"Working on: {p_file.name}...")
    save_folder = output_path / p_file.stem
    
    try:
        result_ds, actual_res, name = process_ptu(p_file, manual_res=TCSPC_RESOLUTION)
        save_outputs(result_ds, actual_res, save_folder, name)
    except Exception as e:
        print(f"  Error processing {p_file.name}: {e}")

print("\n--- All Done ---")

Found 4 files. 
Starting batch processing...
Working on: B16par_10.ptu...
TCSPC channel overflow detected! Max channel: 24147
Channel overflow detected! Max channel: 50
Working on: B16par_10a.ptu...
TCSPC channel overflow detected! Max channel: 24147
Channel overflow detected! Max channel: 50
Working on: B16par_10b.ptu...
TCSPC channel overflow detected! Max channel: 24147
Channel overflow detected! Max channel: 50
Working on: B16par_10c.ptu...
TCSPC channel overflow detected! Max channel: 24147
Channel overflow detected! Max channel: 50

--- All Done ---
